# Pokémon Team-Analyzer - Demo

**PROG2 FS26 - Semester Project**  
Autor: Redon

In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import pandas as pd

from src.api_client import PokeAPIClient, PokeAPIError
from src.pokemon import Pokemon
from src.team import Team
from src.analyzer import TeamAnalyzer
from src.type_chart import TypeChart

# Cache zeigt auf den vom Projekt mitgelieferten Ordner, damit die Demo
# auch ohne Internet läuft. Beim ersten Run mit Netz werden neue Pokemon
# automatisch von der echten PokeAPI nachgeladen.
client = PokeAPIClient(cache_dir=PROJECT_ROOT / "data" / "cache")
print("Setup OK:", PROJECT_ROOT)

## 1. Daten von der PokeAPI laden

Der Client versucht zuerst den Cache, dann erst das Netz, mit Retries und Timeout.

In [ ]:
team_names = ["charizard", "blastoise", "venusaur", "snorlax", "dragonite", "alakazam"]
raw_data = []
for name in team_names:
    try:
        raw_data.append(client.get_pokemon(name))
        print(f"geladen: {name}")
    except PokeAPIError as err:
        print(f"WARNUNG: {name} konnte nicht geladen werden ({err}).")

In [ ]:
# Demonstration der Robustheit: ein Name, den es definitiv nicht gibt.
try:
    client.get_pokemon("nicht-existent-12345")
except PokeAPIError as e:
    print("Erwarteter Fehler abgefangen:", e)

## 2. Domänenobjekte erstellen

Aus den API-Daten werden `Pokemon` -Objekte gebaut und in ein `Team` gefüllt.

In [ ]:
team = Team("Redon's Klassiker", [Pokemon.from_api(data) for data in raw_data])
for p in team:
    print(p)
print("Teamgrösse:", len(team))

## 3. Pandas-Analyse

Das Team wird in einem DataFrame getan und aggregiert.

In [ ]:
analyzer = TeamAnalyzer(team)
df = analyzer.to_stats_dataframe()
df

In [ ]:
# Aggregation: Pandas `agg` mit mehreren Funktionen.
analyzer.summary()

In [ ]:
# Filter-Beispiel mit Pandas: nur Pokemon mit Total > 530.
starke = df[df["total"] > 530].sort_values("total", ascending=False)
starke

## 4. Typ-Coverage des Teams

Welche Angriffstypen sind für das Team gefährlich? Pro Typ wird gezählt, wie viele Teammitglieder schwach / neutral / resistent / immun sind.

In [ ]:
coverage = analyzer.type_coverage()
coverage

In [ ]:
print("Die 5 grössten Schwächen unseres Teams:")
analyzer.biggest_weaknesses()

## 5. Visualisierungen

In [ ]:
analyzer.plot_total_stats()
plt.show()

In [ ]:
analyzer.plot_stats_comparison()
plt.show()

In [ ]:
analyzer.plot_type_coverage_heatmap()
plt.show()

## 6. Vererbung in Aktion: MegaPokemon

`MegaPokemon` erbt von `Pokemon` und überschreibt `total_stats()` mit einem Bonus.

In [ ]:
from src.pokemon import MegaPokemon

charizard = Pokemon.from_api(client.get_pokemon("charizard"))
mega = MegaPokemon(
    name="mega-charizard-x",
    pokedex_id=charizard.pokedex_id,
    types=["fire", "dragon"],
    stats=charizard.stats,
    base_form="charizard",
)
print(f"Normal: {charizard.total_stats()}  |  Mega: {mega.total_stats()}")
print("Mega ist ein Pokemon?", isinstance(mega, Pokemon))

## 7. Tests ausführen

Die Unit-Tests liegen in `tests/`. Sie laufen offline.

In [ ]:
import unittest

loader = unittest.TestLoader()
suite = loader.discover(start_dir=str(PROJECT_ROOT / "tests"), top_level_dir=str(PROJECT_ROOT))
runner = unittest.TextTestRunner(verbosity=1)
result = runner.run(suite)
print(f"\n{result.testsRun} Tests gelaufen, {len(result.failures)} fehlgeschlagen, {len(result.errors)} Fehler.")

## Schlussbemerkungen

- **Komplexität**: `type_coverage()` braucht *O(n)* (n = Teamgröße); die Funktion iteriert genau einmal über die Teammitglieder, die 18 Typen sind eine Konstante.
- **Robustheit**: Netzwerkfehler werden mit Retries abgefangen, Validierung passiert in jedem Konstruktor.
- **Erweiterungsidee**: Eine Methode, die *automatisch* das Pokemon vorschlägt, das die grösste aktuelle Schwäche des Teams entschärft.